# Analisis de Riesgo Academico y Desempeño Estudiantil

Este notebook contiene el analisis exploratorio de datos, visualizaciones y dos estructuras de storytelling analiticas sobre el rendimiento escolar en un dataset de 800,000 estudiantes.

## 1. Carga de Datos y Librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuracion de estilos visuales
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

In [ ]:
file_path = "ulead_visualizacion_800k.csv"
df = pd.read_csv(file_path)
print(f"Dataset cargado con {df.shape[0]:,} filas y {df.shape[1]} columnas.")

## 2. Analisis Exploratorio e Hipotesis

In [ ]:
risk_dist = df['riesgo_academico'].value_counts(normalize=True) * 100
print("Distribucion del riesgo academico (%):")
print(risk_dist)

## 3. Riesgo por Modalidad de Estudio

In [ ]:
mod_risk = pd.crosstab(df['modalidad'], df['riesgo_academico'], normalize='index') * 100
print("Riesgo academico por modalidad (%):")
print(mod_risk)

## 4. Visualizacion de Soporte

In [ ]:
# Grafico 1: Riesgo Academico por Modalidad
plt.figure(figsize=(8, 5))
colors = ['#ff6b6b' if x in ['Virtual', 'Hibrida'] else '#4dabf7' for x in mod_risk.index]
sns.barplot(x=mod_risk.index, y=mod_risk['Alto'], palette=colors)
plt.title('Porcentaje de Alumnos en Riesgo Alto por Modalidad')
plt.ylabel('% Riesgo Alto')
plt.xlabel('Modalidad')
for i, val in enumerate(mod_risk['Alto']):
    plt.text(i, val + 0.3, f"{val:.2f}%", ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('riesgo_por_modalidad.png', dpi=150)
plt.show()

In [ ]:
# Grafico 2: Asistencia promedio por riesgo y modalidad
att_stats = df.groupby(['modalidad', 'riesgo_academico'])['asistencia'].mean().reset_index()
plt.figure(figsize=(10, 6))
sns.barplot(data=att_stats, x='modalidad', y='asistencia', hue='riesgo_academico', palette='coolwarm')
plt.title('Asistencia Promedio por Modalidad y Nivel de Riesgo')
plt.ylabel('% Asistencia Promedio')
plt.xlabel('Modalidad')
plt.legend(title='Riesgo Academico')
plt.tight_layout()
plt.savefig('asistencia_riesgo_modalidad.png', dpi=150)
plt.show()

## 5. Correlacion con la Nota Final

In [ ]:
numeric_cols = ['horas_estudio', 'asistencia', 'participacion', 'tareas_entregadas', 'uso_plataforma', 'nota_final']
corr_matrix = df[numeric_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='Blues', fmt='.2f')
plt.title('Matriz de Correlacion de Desempeño Estudiantil')
plt.tight_layout()
plt.savefig('matriz_correlacion.png', dpi=150)
plt.show()

## 6. Respuestas a las 10 Preguntas de Analisis (Cuestionario)

### Q1: ¿Como se distribuye la nota final?
El promedio general es de 93.20 con desviacion de 7.50, sesgada a la izquierda (calificaciones altas) con mediana de 95.

In [ ]:
df['nota_final'].describe()

### Q2: ¿Hay diferencias de nota final entre modalidades?
La nota final promedio es levemente inferior en la modalidad virtual (92.74) frente a hibrida (93.55) y presencial (93.30), con mayor dispersion en la modalidad virtual (desviacion de 7.72).

In [ ]:
df.groupby('modalidad')['nota_final'].agg(['mean', 'std'])

### Q3: ¿Que carrera tiene mayor promedio de nota final?
Ciberseguridad lidera con un promedio de 93.23, seguida de cerca por Administracion con 93.20.

In [ ]:
df.groupby('carrera')['nota_final'].mean().sort_values(ascending=False)

### Q4: ¿Que carrera tiene mayor variabilidad en las notas?
Finanzas presenta la mayor variabilidad con una desviacion estandar de 7.52.

In [ ]:
df.groupby('carrera')['nota_final'].std().sort_values(ascending=False)

### Q5: ¿Existe relacion entre horas de estudio y nota final?
Si, existe una relacion lineal positiva moderada-fuerte con una correlacion de 0.5960.

In [ ]:
print(f"Correlacion horas-nota: {df['horas_estudio'].corr(df['nota_final']):.4f}")

### Q6: ¿Existe relacion entre asistencia y nota final?
Si, hay una correlacion positiva debil de 0.2154, aunque es el principal factor de de-engagement de alumnos en riesgo.

In [ ]:
print(f"Correlacion asistencia-nota: {df['asistencia'].corr(df['nota_final']):.4f}")

### Q7: ¿Los estudiantes que trabajan tienen menor rendimiento?
Si. Los alumnos que trabajan promedian 89.37 frente a los que no trabajan que promedian 96.32, una diferencia de casi 7 puntos.

In [ ]:
df.groupby('trabaja')['nota_final'].agg(['mean', 'std'])

### Q8: ¿Que campus concentra mayor riesgo academico?
El campus Virtual concentra el mayor indice de riesgo alto con un 12.93% (34,410 alumnos).

In [ ]:
pd.crosstab(df['campus'], df['riesgo_academico'], normalize='index') * 100

### Q9: ¿El uso de plataforma se asocia con mejores notas?
La relacion lineal es muy debil (correlacion de 0.0616), pero segmentando por cuartiles de uso se nota un aumento progresivo de 92.57 a 93.78.

In [ ]:
print(f"Correlacion plataforma-nota: {df['uso_plataforma'].corr(df['nota_final']):.4f}")

### Q10: Recomendaciones finales para ULEAD
1. Desplegar sistema de alertas SATA ante caidas de asistencia abajo del 75% en clases virtuales.
2. Focalizar el presupuesto de tutoria en alumnos no presenciales (hibrido y virtual).
3. Flexibilizar entregas y dar tutorias asincronas fines de semana para alumnos que trabajan.

## 7. Storytelling 2: La Brecha del Estudiante Trabajador (Formato Estructurado)

1. **¿Que problema estamos intentando resolver?** Mitigar la brecha de calificaciones y el riesgo de bajo rendimiento en el segmento de estudiantes que trabajan, quienes promedian calificaciones mas bajas y muestran mayor variabilidad.
2. **¿Que patron importante aparecio?** Los estudiantes que no trabajan promedian 96.32, mientras que los estudiantes que si trabajan promedian 89.37. Existe una brecha directa de 6.95 puntos porcentuales de rendimiento escolar, con mayor variabilidad (Std de 7.99 vs 5.32) en el grupo trabajador.
3. **¿Que grafico respalda el hallazgo?** El diagrama de caja comparativo (Boxplot) de notas finales segun estado laboral.
4. **¿Por que importa?** Demuestra que combinar trabajo y estudio desgasta el rendimiento academico promedio e incrementa la dispersion de calificaciones, afectando la aprobacion de becas y permanencia escolar, a pesar de reportar horas de estudio semanales equivalentes.
5. **¿Que recomendamos hacer ahora?** Implementar politicas de gracia automatica de 48 horas en entregas para estudiantes activos que laboran, y facilitar canales de tutorias asincronas fines de semana.